In [1]:
import scipy.stats as stats
import pandas as pd
import numpy as np
from statsmodels.stats.multitest import multipletests
import json
import warnings

warnings.filterwarnings(
    "ignore",
    message="Workbook contains no default style, apply openpyxl's default"
)

s_a_pathway = pd.read_excel('../data/suger acid kegg pathways.xlsx', header=None)
sugar_kos = s_a_pathway[s_a_pathway[1]=='sugar'][0].tolist()
acid_kos = s_a_pathway[s_a_pathway[1]=='acid'][0].tolist()
s_a_type = pd.read_csv('../data/suger_acid_type.csv')
gene_sets = {}

for sugar in s_a_type[s_a_type['Category']=='Sugar']['Object'].tolist():
    path = '../data/suger_acid_KAGs/' + sugar + '_KAG.json'
    with open(path, encoding='utf-8') as f:
        hif_dict = json.load(f)
    pos_hif = list(hif_dict['pos_hif'].keys())
    gene_sets[sugar] = pos_hif

def perform_enrichment_test(set_A, list_X, total_background_elements):
    set_X = set(list_X)
    intersection_count = len(set_A.intersection(set_X))
    set_A_size = len(set_A)
    list_X_size = len(set_X)
    
    p_value = stats.hypergeom.sf(
        intersection_count - 1, 
        total_background_elements, 
        set_A_size, 
        list_X_size
    )
    return p_value

raw_p_values_data = []

for compound, hif_genes in gene_sets.items():
    gene_set = set(hif_genes)
    with open(f'/Kun_data/GELATO/gelato/vocab/{compound}_w2i.json', encoding='utf-8') as f:
        total_background_elements = len(json.load(f))
    p_val_A = perform_enrichment_test(gene_set, sugar_kos, total_background_elements)
    raw_p_values_data.append({'Compound_name': compound, 'Pathway': 'Sugar Pathway', 'raw_p_value': p_val_A})
    
    p_val_B = perform_enrichment_test(gene_set, acid_kos, total_background_elements)
    raw_p_values_data.append({'Compound_name': compound, 'Pathway': 'Acid Pathway', 'raw_p_value': p_val_B})

df_results = pd.DataFrame(raw_p_values_data)
print(df_results)

p_values = df_results['raw_p_value'].tolist()
reject, pvals_corrected, _, _ = multipletests(p_values, alpha=0.05, method='fdr_bh')

df_results['fdr_q_value'] = pvals_corrected
df_results['is_significant_fdr'] = reject

print(df_results)

significant_results_fdr = df_results[df_results['is_significant_fdr']].copy()


   Compound_name        Pathway    raw_p_value
0        maltose  Sugar Pathway  3.741970e-191
1        maltose   Acid Pathway   6.301679e-05
2        sucrose  Sugar Pathway  1.959850e-117
3        sucrose   Acid Pathway   1.922183e-05
4        glucose  Sugar Pathway   3.508781e-21
5        glucose   Acid Pathway   6.674020e-07
6      raffinose  Sugar Pathway   1.109163e-64
7      raffinose   Acid Pathway   5.388751e-06
8        lactose  Sugar Pathway  1.015668e-102
9        lactose   Acid Pathway   2.907778e-04
10    cellobiose  Sugar Pathway   1.576572e-67
11    cellobiose   Acid Pathway   3.161750e-01
12     trehalose  Sugar Pathway   2.075133e-95
13     trehalose   Acid Pathway   3.277905e-01
14      D-ribose  Sugar Pathway   1.570520e-40
15      D-ribose   Acid Pathway   1.460868e-04
16      fructose  Sugar Pathway   2.843968e-07
17      fructose   Acid Pathway   6.009696e-06
18     melibiose  Sugar Pathway   7.018036e-95
19     melibiose   Acid Pathway   3.136121e-01
20       mann

In [2]:
gene_sets = {}

for acid in s_a_type[s_a_type['Category']=='Acid']['Object'].tolist():
    path = '../data/suger_acid_KAGs/' + acid + '_KAG.json'
    with open(path, encoding='utf-8') as f:
        hif_dict = json.load(f)
    pos_hif = list(hif_dict['pos_hif'].keys())
    gene_sets[acid] = pos_hif

def perform_enrichment_test(set_A, list_X, total_background_elements):
    set_X = set(list_X)
    intersection_count = len(set_A.intersection(set_X))
    set_A_size = len(set_A)
    list_X_size = len(set_X)
    
    p_value = stats.hypergeom.sf(
        intersection_count - 1, 
        total_background_elements, 
        set_A_size, 
        list_X_size
    )
    return p_value

raw_p_values_data = []

for compound, hif_genes in gene_sets.items():
    gene_set = set(hif_genes)
    with open(f'/Kun_data/GELATO/gelato/vocab/{compound}_w2i.json', encoding='utf-8') as f:
        total_background_elements = len(json.load(f))
    p_val_A = perform_enrichment_test(gene_set, sugar_kos, total_background_elements)
    raw_p_values_data.append({'Compound_name': compound, 'Pathway': 'Sugar Pathway', 'raw_p_value': p_val_A})
    
    p_val_B = perform_enrichment_test(gene_set, acid_kos, total_background_elements)
    raw_p_values_data.append({'Compound_name': compound, 'Pathway': 'Acid Pathway', 'raw_p_value': p_val_B})

df_results = pd.DataFrame(raw_p_values_data)
print(df_results)

p_values = df_results['raw_p_value'].tolist()
reject, pvals_corrected, _, _ = multipletests(p_values, alpha=0.05, method='fdr_bh')

df_results['fdr_q_value'] = pvals_corrected
df_results['is_significant_fdr'] = reject

print(df_results)

significant_results_fdr = df_results[df_results['is_significant_fdr']].copy()


   Compound_name        Pathway   raw_p_value
0        citrate  Sugar Pathway  2.390335e-05
1        citrate   Acid Pathway  3.170576e-36
2         malate  Sugar Pathway  1.347541e-01
3         malate   Acid Pathway  9.999036e-12
4      gluconate  Sugar Pathway  4.687792e-48
5      gluconate   Acid Pathway  2.987528e-05
6        adipate  Sugar Pathway  9.833824e-01
7        adipate   Acid Pathway  2.998062e-33
8      hippurate  Sugar Pathway  2.098128e-01
9      hippurate   Acid Pathway  1.767251e-12
10       acetate  Sugar Pathway  9.999154e-01
11       acetate   Acid Pathway  6.673282e-17
12       lactate  Sugar Pathway  9.762347e-01
13       lactate   Acid Pathway  1.298152e-14
14    propionate  Sugar Pathway  9.986600e-01
15    propionate   Acid Pathway  5.107464e-23
16     succinate  Sugar Pathway  2.001119e-01
17     succinate   Acid Pathway  4.771719e-15
18      malonate  Sugar Pathway  1.108956e-04
19      malonate   Acid Pathway  1.227544e-05
20       formate  Sugar Pathway  9